<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 2</b>

## Unit 3: CrewAI — Agents, Tasks, and Crews

**CV Raman Global University, Bhubaneswar**
*AI Center of Excellence*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Understand CrewAI's core concepts** — Agent, Task, and Crew
2. **Build your first crew** — create an agent, define a task, and run it with `kickoff()`
3. **Define specialized roles** — build multi-agent crews with distinct expertise
4. **Use execution modes** — sequential processing and agent pipelines
5. **Switch LLM providers** — use any model via CrewAI's built-in LiteLLM support

## 1. Environment Setup

In [ ]:
!pip install -q crewai

In [ ]:
import os
from getpass import getpass
from IPython.display import Markdown, display
from crewai import Agent, Task, Crew, Process, LLM

print("Enter your API keys (press Enter to skip optional ones):\n")

google_key = getpass("Google Gemini API Key (recommended): ")
if google_key:
    os.environ['GEMINI_API_KEY'] = google_key
    print("Gemini key configured")

openai_key = getpass("OpenAI API Key (optional): ")
if openai_key:
    os.environ['OPENAI_API_KEY'] = openai_key
    print("OpenAI key configured")

groq_key = getpass("Groq API Key (optional): ")
if groq_key:
    os.environ['GROQ_API_KEY'] = groq_key
    print("Groq key configured")

In [ ]:
# CrewAI's LLM class uses LiteLLM under the hood
# Use provider/model format — same pattern for any provider
# llm = LLM(model="gemini/gemini-2.0-flash")

# To use a different provider, just change the model string:
llm = LLM(model="openai/gpt-4o-mini")
# llm = LLM(model="groq/llama-3.3-70b-versatile")

print(f"LLM configured: {llm.model}")

---

## 2. The Problem — Why Multi-Agent Frameworks?

In Unit 2, we built agents with the OpenAI Agents SDK — great for tool use and handoffs. But what if you need a **team of specialized agents** working together on a structured workflow?

**CrewAI** is a Python framework designed exactly for this. It organizes work around three simple concepts:

| DIY Multi-Agent | CrewAI |
|---|---|
| Define agent roles in prompt strings | `Agent(role, goal, backstory)` |
| Manually pass outputs between agents | Tasks auto-chain via `Process.sequential` |
| Write your own orchestration loop | `Crew.kickoff()` handles everything |
| Track which agent does what | Agents assigned to specific tasks |

The three core building blocks:

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│     Agent        │     │     Task         │     │     Crew         │
├─────────────────┤     ├─────────────────┤     ├─────────────────┤
│ role             │     │ description      │     │ agents           │
│ goal             │     │ expected_output  │     │ tasks            │
│ backstory        │     │ agent            │     │ process          │
│ llm              │     └─────────────────┘     └────────┬────────┘
└─────────────────┘                                       │
                                                  crew.kickoff()
                                                          │
                                                          ▼
                                                    Final Output
```

---

## 3. Your First Agent and Crew

An **Agent** has three key properties that shape its behavior:
- **role** — what it does (e.g., "Researcher")
- **goal** — what it's trying to achieve
- **backstory** — personality and expertise that guides behavior

A **Task** defines the work: what needs to be done and what the output should look like.

A **Crew** brings agents and tasks together and runs them with `kickoff()`.

In [ ]:
# Create a researcher agent
researcher = Agent(
    role="Researcher",
    goal="Find accurate and useful information on any topic",
    backstory="You are a skilled researcher who excels at finding and summarizing information clearly.",
    llm=llm,
    verbose=True
)

print(f"Agent created — Role: {researcher.role}")

In [ ]:
# Define a task and assign it to the researcher
research_task = Task(
    description="Research the main applications of AI in healthcare. List 3 key applications with brief explanations.",
    expected_output="A list of 3 AI applications in healthcare with one-sentence explanations for each.",
    agent=researcher
)

# Build a crew with one agent and one task
crew = Crew(
    agents=[researcher],
    tasks=[research_task],
    verbose=True
)

print(f"Crew created — Agents: {len(crew.agents)}, Tasks: {len(crew.tasks)}")

In [ ]:
# Run the crew
result = crew.kickoff()

print("\nFinal Output:")
print("-" * 50)
display(Markdown(str(result)))

> **The pattern:** Create agents → Define tasks → Build crew → `kickoff()`. Every CrewAI project follows this flow.

---

## 4. Specialized Roles — Multi-Agent Crews

A single agent can only do so much. Real workflows need **specialists** — each agent with a distinct role, like a team:

```
┌──────────────────┐          ┌──────────────────┐
│  Research         │          │  Content Writer   │
│  Specialist       │  ────>   │                  │
│                  │  output   │  Uses research   │
│  Finds facts     │  flows    │  to write article │
└──────────────────┘          └──────────────────┘
```

The key insight: **role, goal, and backstory** together define how an agent behaves. A researcher with a research-focused backstory produces different output than a writer with a writing-focused backstory, even when using the same LLM.

In [ ]:
# Create two specialized agents
researcher = Agent(
    role="Research Specialist",
    goal="Find accurate, relevant, and comprehensive information on any topic",
    backstory="""You are an experienced researcher with a talent for finding 
    key information quickly. You focus on facts and evidence, always looking 
    for reliable sources. You present your findings in a clear, organized manner.""",
    llm=llm,
    verbose=True
)

writer = Agent(
    role="Content Writer",
    goal="Transform research into clear, engaging, and well-structured content",
    backstory="""You are a skilled writer who excels at making complex topics 
    easy to understand. You write in a clear, conversational style that engages 
    readers. You always structure your content with good formatting.""",
    llm=llm,
    verbose=True
)

print(f"Agents: {researcher.role}, {writer.role}")

In [ ]:
# Assign tasks to matching specialists
research_task = Task(
    description="""Research the topic of 'sustainable living practices'. 
    Find 4-5 key practices that individuals can adopt. 
    Include a brief explanation of why each practice matters.""",
    expected_output="A list of 4-5 sustainable living practices with explanations.",
    agent=researcher  # Research task → researcher
)

writing_task = Task(
    description="""Using the research provided, write a short blog post 
    about sustainable living. Make it engaging and actionable. 
    Keep it under 300 words.""",
    expected_output="A short, engaging blog post about sustainable living (under 300 words).",
    agent=writer  # Writing task → writer
)

# Build and run the 2-agent crew
content_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    verbose=True
)

result = content_crew.kickoff()

print("\nFinal Blog Post:")
print("-" * 50)
display(Markdown(str(result)))

---

## 5. Execution Modes

CrewAI supports different ways for agents to work together:

### Sequential vs Hierarchical

| Mode | How It Works | Use When |
|---|---|---|
| `Process.sequential` | Tasks run one after another, output flows to the next | Tasks depend on each other |
| `Process.hierarchical` | A manager agent delegates and coordinates | Complex workflows needing dynamic routing |

Sequential is the default and most common mode. Let's see it in action explicitly.

### Sequential Execution

Tasks run in order. Each task's output automatically becomes context for the next task.

```
Task 1 (Researcher)  ──>  Task 2 (Writer)
  output flows as            uses research
  context to next            to write article
```

In [ ]:
# Explicitly set process=Process.sequential
research_task = Task(
    description="Research 3 key facts about solar energy.",
    expected_output="A list of 3 key facts about solar energy.",
    agent=researcher
)

write_task = Task(
    description="Based on the research, write a 2-sentence summary about solar energy.",
    expected_output="A 2-sentence summary about solar energy.",
    agent=writer
)

sequential_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,  # Explicit — tasks run one by one
    verbose=True
)

result = sequential_crew.kickoff()

print("\nSequential Result:")
display(Markdown(str(result)))

### Agent Pipeline — Chaining Multiple Specialists

Pipelines chain 3+ agents where each builds on the previous output:

```
Researcher  ──>  Analyst  ──>  Summarizer
 gathers          finds         creates
 raw facts        insights      takeaway
```

In [ ]:
# Three-agent pipeline: Research → Analyze → Summarize
researcher = Agent(
    role="Researcher",
    goal="Gather raw information",
    backstory="You find facts quickly and present them clearly.",
    llm=llm,
    verbose=True
)

analyst = Agent(
    role="Analyst",
    goal="Extract key insights from information",
    backstory="You identify the most important points from raw data.",
    llm=llm,
    verbose=True
)

summarizer = Agent(
    role="Summarizer",
    goal="Create brief, actionable summaries",
    backstory="You write concise summaries that capture the essence.",
    llm=llm,
    verbose=True
)

print("Three pipeline agents created!")

In [ ]:
# Tasks form a pipeline — each builds on the previous
task1 = Task(
    description="Find 4 facts about electric vehicles.",
    expected_output="4 facts about electric vehicles.",
    agent=researcher
)

task2 = Task(
    description="From the research, identify the 2 most impactful facts and explain why.",
    expected_output="2 most impactful facts with explanations.",
    agent=analyst
)

task3 = Task(
    description="Write a one-sentence takeaway based on the analysis.",
    expected_output="One-sentence takeaway about electric vehicles.",
    agent=summarizer
)

pipeline_crew = Crew(
    agents=[researcher, analyst, summarizer],
    tasks=[task1, task2, task3],
    process=Process.sequential,
    verbose=True
)

result = pipeline_crew.kickoff()

print("\nFinal Takeaway:")
display(Markdown(str(result)))

---

## 6. Key Takeaways

### Concept Map

```
Agent(role, goal, backstory, llm)  ─── defines a specialist
  │
  └── assigned to ──> Task(description, expected_output, agent)
                        │
                        └── grouped into ──> Crew(agents, tasks, process)
                                              │
                                              └── crew.kickoff() → Final Output

Process.sequential:  Task1 ──> Task2 ──> Task3  (output flows automatically)
```

### Quick Reference

| Concept | Code | When to Use |
|---|---|---|
| **Create agent** | `Agent(role, goal, backstory, llm)` | Always — the basic building block |
| **Define task** | `Task(description, expected_output, agent)` | Define a specific piece of work |
| **Build crew** | `Crew(agents, tasks, process)` | Bring agents and tasks together |
| **Run crew** | `crew.kickoff()` | Execute the workflow |
| **Sequential** | `process=Process.sequential` | Tasks depend on each other |
| **Switch LLM** | `LLM(model="provider/model")` | Use any LiteLLM-supported provider |

---

## 7. Exercises

### Exercise 1: Your Own Crew (Beginner)
Create a single-agent crew on a topic of your choice (e.g., space exploration, Indian cuisine, cricket). Define a clear role, goal, and backstory. Run it and observe how the backstory influences the output.

### Exercise 2: Three-Agent Pipeline (Intermediate)
Build a pipeline with three agents: **Brainstormer** (generates ideas), **Evaluator** (ranks them), **Presenter** (formats the best idea as a pitch). Test it with: *"New mobile app ideas for college students."*

### Exercise 3: Switch LLM Providers (Intermediate)
Take the 2-agent crew from Section 4 and run it with a different LLM provider (e.g., `groq/llama-3.3-70b-versatile` or `openai/gpt-4o-mini`). Compare the output quality and speed. Try using different providers for different agents in the same crew.

---

**What's Next?** In the next notebook, we'll extend agents with custom tools, add memory, and build a complete research crew project.

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 2
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---